# E4. Concellos da Coruña (Web Scraping)
## 1. Preparación e importaciones
Aquí importamos las librerías que vimos en clase (`requests`, `BeautifulSoup`, `pandas`, `Path`, etc.). También dejamos configuradas las URLs de Wikipedia, nuestro `User-Agent` para identificarnos y las rutas de las carpetas `raw` y `clean` (con `mkdir` se crean solas si no existen).

In [50]:
from io import StringIO
from pathlib import Path
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup

# Datos básicos de la web y nuestra identificación
URL_BASE = "https://en.wikipedia.org/"
URL_PAXINA = "https://en.wikipedia.org/wiki/List_of_municipalities_in_A_Coru%C3%B1a"
CABECEIRAS = {"User-Agent": "exercicio-e4-sbd/1.0 (carlos@exemplo.com)"}

# Carpetas para guardar los archivos (se crean solas si no existen)
RAW = Path("../datos/raw")
CLEAN = Path("../datos/clean")
RAW.mkdir(parents=True, exist_ok=True)
CLEAN.mkdir(parents=True, exist_ok=True)

## 2. Comprobar permisos con `robots.txt`
Antes de descargar nada, miramos si Wikipedia nos da permiso. Como Wikipedia bloquea `robots.read()` si vas sin `User-Agent`, descargamos primero el archivo `robots.txt` con `requests` pasándole nuestras cabeceras y luego se lo damos a `RobotFileParser`.

In [51]:
# Descargamos el robots.txt con nuestra cabecera para que Wikipedia nos deje leerlo
url_robots = urljoin(URL_BASE, "robots.txt")
resposta_robots = requests.get(url_robots, headers=CABECEIRAS, timeout=10)

# Le pasamos las líneas al analizador de robots
robots = RobotFileParser()
robots.parse(resposta_robots.text.splitlines())

# Comprobamos si tenemos permiso para entrar en la página de los concellos
permitido = robots.can_fetch(CABECEIRAS["User-Agent"], URL_PAXINA)
print("¿Está permitido descargar la página?:", permitido)

¿Está permitido descargar la página?: True


### Sobre la licencia y los datos personales
* **Licencia (CC BY-SA 4.0):** 
Nos deja descargar, usar y modificar los datos sin problema (incluso para uso comercial), siempre y cuando citemos que vienen de la Wikipedia y mantengamos la misma licencia si compartimos el resultado.
* **Datos personales:** 
No hay ningún dato personal ni privado, solo son nombres de ayuntamientos y el número total de habitantes de cada uno.

## 3. Descarga de la página en `datos/raw`
Usamos la misma función que vimos en el ejemplo de la lonja[cite: 2]: primero comprueba con `.exists()` si ya tenemos el HTML guardado en `raw` para no molestar al servidor descargándolo mil veces. Si no está, lo descarga y guarda los bytes originales tal cual llegan.

In [52]:

def descargar(url, destino):
    if destino.exists():
        return "xa existía"
    r = requests.get(url, headers=CABECEIRAS, timeout=10)
    r.raise_for_status()
    destino.write_bytes(r.content)
    return "descargado"

ficheiro_html = RAW / "concellos_coruna.html"
estado = descargar(URL_PAXINA, ficheiro_html)
print(ficheiro_html.name, "->", estado)

concellos_coruna.html -> xa existía


## 4. Extraer la tabla con BeautifulSoup
Leemos el archivo HTML desde `raw` y buscamos la tabla fijándonos en que sus cabeceras (`th`) tengan "name" y "population", así no dependemos de en qué posición esté[cite: 1, 5]. Luego recorremos las filas (`tr`) y celdas (`td`) guardando cada concello en una lista de diccionarios (saltándonos la fila final del "Total").

In [53]:
# Leemos los bytes del archivo guardado
sopa = BeautifulSoup(ficheiro_html.read_bytes(), "html.parser")

# Buscamos la tabla por el texto de sus cabeceras, no por su posición
taboa_concellos = None
for t in sopa.find_all("table"):
    cabeceiras = [th.get_text(strip=True).lower() for th in t.find_all("th")]
    if "name" in cabeceiras and any("population" in c for c in cabeceiras):
        taboa_concellos = t
        break

# Recorremos las filas (tr) y guardamos concello y población en una lista de diccionarios
filas = []
for tr in taboa_concellos.find_all("tr"):
    celas = tr.find_all("td")
    # Solo cogemos filas de datos normales (evitamos cabeceras y la fila final del total)
    if len(celas) >= 2:
        nome = celas[0].get_text(strip=True)
        poboacion = celas[1].get_text(strip=True)
        if nome.lower() != "total":
            filas.append({
                "concello": nome,
                "poboacion": poboacion
            })

df_concellos = pd.DataFrame(filas)
print("Tamaño de la tabla extraída:", df_concellos.shape)
df_concellos.head()

Tamaño de la tabla extraída: (93, 2)


,concello,poboacion
0,Abegondo,"5,578"
1,Ames,"32,812"
2,Aranga,"1,799"
3,Ares,"6,156"
4,Arteixo,"34,038"


## 5. Limpieza de los datos de población
Antes de pasar la población a número entero (`int`), comprobamos con una expresión regular si hay algún número que no siga el formato inglés de Wikipedia (números separados por comas).

In [54]:
# Miramos qué filas tienen cosas raras (algo que no sean solo números y comas de miles)
formato_raro = df_concellos[~df_concellos["poboacion"].str.match(r"^[\d,]+$")]
print("Filas con formato diferente antes de limpiar:")
print(formato_raro)

# Limpiamos quitando notas entre corchetes, comas de miles y puntos
texto_limpo = (
    df_concellos["poboacion"]
    .str.replace(r"\[.*?\]", "", regex=True)
    .str.replace(",", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.strip()
)

# Convertimos a número entero
df_concellos["poboacion"] = pd.to_numeric(texto_limpo).astype(int)
print("\nTipos de datos tras la limpieza:")
print(df_concellos.dtypes)

Filas con formato diferente antes de limpiar:
   concello poboacion
9  Boimorto     1.801

Tipos de datos tras la limpieza:
concello       str
poboacion    int64
dtype: object


**¿Qué hicimos con los valores raros?**
Al comprobar el formato vimos que **Boimorto** tenía un punto (`1.801`) en lugar de una coma como el resto de concellos. Para dejarlo todo limpio, quitamos tanto las comas como los puntos (y posibles referencias entre corchetes `[...]`) antes de convertir la columna a número entero (`int64`).

## 6. Validación de los datos
Aquí comprobamos tres cosas para estar seguros de que no metimos la pata:
1. Que tenemos exactamente los 93 concellos y ninguno duplicado.
2. Que la suma de nuestra columna `poboacion` da exactamente lo mismo que el total provincial que aparece escrito en el texto de la página (`1,128,320`).
3. Qué pasa si leemos la tabla directamente con el atajo `pd.read_html`.

In [55]:
import re

# Comprobar que hay 93 concellos y ningún repetido
num_concellos = len(df_concellos)
repetidos = df_concellos["concello"].duplicated().sum()
print("Número de concellos (debe ser 93):", num_concellos)
print("Concellos repetidos (debe ser 0):", repetidos)

# Extraer el total de la página y comparar con nuestra suma
texto_paxina = sopa.get_text()
# Buscamos el número total de habitantes en el texto
coincidencias = re.findall(r"1,\d{3},\d{3}", texto_paxina)
total_paxina = int(coincidencias[0].replace(",", ""))
suma_taboa = df_concellos["poboacion"].sum()

print("Total que dice la página:", total_paxina)
print("Suma de nuestra tabla   :", suma_taboa)
print("¿Coinciden los totales? :", total_paxina == suma_taboa)

# Leer
taboas_pandas = pd.read_html(StringIO(ficheiro_html.read_text(encoding="utf-8")))
df_read_html = taboas_pandas[0]
print("\nForma de la tabla con read_html:", df_read_html.shape)
df_read_html.head()

Número de concellos (debe ser 93): 93
Concellos repetidos (debe ser 0): 0
Total que dice la página: 1128320
Suma de nuestra tabla   : 1128320
¿Coinciden los totales? : True

Forma de la tabla con read_html: (93, 2)


,Name,Population (2024)[20]
0,Abegondo,5578.0
1,Ames,32812.0
2,Aranga,1799.0
3,Ares,6156.0
4,Arteixo,34038.0


**Diferencias con `pd.read_html`:**
Al usar `pd.read_html`, Pandas deja el nombre sucio en la columna (`Population (2024)[20]`) y, como Boimorto tenía un punto (`1.801`), Pandas se cree que es un número con decimales y convierte toda la columna a `float` (por eso salen con `.0` al final y Boimorto queda como si tuviera menos de 2 habitantes). Haciéndolo nosotros con BeautifulSoup controlamos la limpieza antes de convertir a número.

## 7. Preguntas sobre la población
### Pregunta 1: ¿Cuáles son los cinco concellos más poblados?

In [56]:
cinco_mais_poboados = df_concellos.sort_values("poboacion", ascending=False).head(5)
print("1. Cinco concellos más poblados:")
print(cinco_mais_poboados)

1. Cinco concellos más poblados:
                  concello  poboacion
29                A Coruña     249261
77  Santiago de Compostela      99536
35                  Ferrol      64218
53                   Narón      39285
57                 Oleiros      38333


**Interpretación:** 
Los 5 concellos más poblados son **A Coruña** (249.261 hab.), **Santiago de Compostela** (99.536 hab.), **Ferrol** (64.218 hab.), **Narón** (39.285 hab.) y **Oleiros** (38.333 hab.). Se ve claramente que la población se concentra en las tres grandes ciudades y en los ayuntamientos pegados a ellas.

### Pregunta 2: ¿Cuántos concellos tienen menos de 5.000 habitantes y qué porcentaje representan?

In [57]:
menos_de_5000 = df_concellos[df_concellos["poboacion"] < 5000]
cant_menos_5000 = len(menos_de_5000)
porcentaxe_menos_5000 = round((cant_menos_5000 / len(df_concellos)) * 100, 2)

print(f"Concellos con menos de 5.000 hab.: {cant_menos_5000} de {len(df_concellos)} ({porcentaxe_menos_5000}%)")

Concellos con menos de 5.000 hab.: 43 de 93 (46.24%)


**Interpretación:** 
Hay **43 concellos** que no llegan a los 5.000 habitantes, lo que supone casi la mitad de todos los ayuntamientos de la provincia (**un 46,24%**), demostrando que hay muchísimos municipios pequeños y rurales.

### Pregunta 3: ¿Qué porcentaje de la población provincial vive en los tres concellos más poblados?

In [58]:
tres_mais_poboados = df_concellos.sort_values("poboacion", ascending=False).head(3)
suma_tres_primeiros = tres_mais_poboados["poboacion"].sum()
porcentaxe_tres_primeiros = round((suma_tres_primeiros / suma_taboa) * 100, 2)

print("Los 3 concellos más poblados:")
print(tres_mais_poboados)
print(f"\nSuma de los 3 concellos: {suma_tres_primeiros} habitantes")
print(f"Porcentaje sobre el total provincial: {porcentaxe_tres_primeiros}%")

Los 3 concellos más poblados:
                  concello  poboacion
29                A Coruña     249261
77  Santiago de Compostela      99536
35                  Ferrol      64218

Suma de los 3 concellos: 413015 habitantes
Porcentaje sobre el total provincial: 36.6%


**Interpretación:**
 En los tres concellos más grandes (**A Coruña, Santiago de Compostela y Ferrol**) viven en total 413.015 personas, lo que significa que más de un tercio de toda la provincia (**el 36,6%**) vive solo en esas tres ciudades.

### Pregunta 4: ¿Cuántos concellos hacen falta para reunir la mitad de la población de la provincia?

In [59]:
df_ordenado = df_concellos.sort_values("poboacion", ascending=False).reset_index(drop=True)
df_ordenado["acumulado"] = df_ordenado["poboacion"].cumsum()
metade_poboacion = suma_taboa / 2
concellos_para_metade = int((df_ordenado["acumulado"] < metade_poboacion).sum() + 1)

print(f"Mitad de la población provincial: {metade_poboacion:,.0f} habitantes")
print(f"Concellos necesarios: {concellos_para_metade}\n")
print(df_ordenado.head(concellos_para_metade))

Mitad de la población provincial: 564,160 habitantes
Concellos necesarios: 8

                 concello  poboacion  acumulado
0                A Coruña     249261     249261
1  Santiago de Compostela      99536     348797
2                  Ferrol      64218     413015
3                   Narón      39285     452300
4                 Oleiros      38333     490633
5                 Arteixo      34038     524671
6                    Ames      32812     557483
7                Carballo      31595     589078


**Interpretación:** 
Solo hacen falta **8 concellos** (de los 93 que hay) para superar la mitad de la población de toda la provincia (A Coruña, Santiago, Ferrol, Narón, Oleiros, Arteixo, Ames y Carballo).

## 8. Guardar los datos limpios
Guardamos la tabla final ya limpia en formato CSV con codificación UTF-8 dentro de la carpeta `datos/clean/`.

In [60]:
ruta_csv = CLEAN / "concellos_coruna.csv"
df_concellos.to_csv(ruta_csv, index=False, encoding="utf-8")
print("Archivo guardado correctamente en:", ruta_csv)

Archivo guardado correctamente en: ../datos/clean/concellos_coruna.csv


## 9. Uso de la inteligencia artificial
Se ha utilizado IA (Gemini) como apoyo para consultar dudas sobre expresiones regulares en la limpieza de strings, revisar la estructura de extracción con `BeautifulSoup` y dar formato a las explicaciones del cuaderno.